In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/pre_cell_2.pickle

In [ ]:
%%PandasProfile
### cell 2 ###

# Optimized cell 2
import os
import re

input_dir = '/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'
mapping = {'Number of Record': 'Number of Records'}
year_re = re.compile(r'year_(\d+)_quarter')
quarter_re = re.compile(r'quarter_(\d+)\.csv')

mobile_dfs = []
broadband_dfs = []

for root, _, files in os.walk(input_dir):
    for fname in files:
        path = os.path.join(root, fname)
        df = pd.read_csv(path, thousands=',').convert_dtypes()
        # apply any column corrections
        df.rename(columns=mapping, inplace=True)
        # extract year and quarter once per file
        yr = year_re.search(fname).group(1)
        qt = quarter_re.search(fname).group(1)
        df['Year'] = yr
        df['Quarter'] = qt
        # collect by type
        if 'mobile' in fname:
            mobile_dfs.append(df)
        else:
            broadband_dfs.append(df)

# concatenate once
Mobile_df = pd.concat(mobile_dfs, ignore_index=False) if mobile_dfs else pd.DataFrame([], columns=data.columns)
Broadband_df = pd.concat(broadband_dfs, ignore_index=False) if broadband_dfs else pd.DataFrame([], columns=data.columns)

print(Broadband_df.shape)
print(Mobile_df.shape)

# final type casting and sorting
for df in (Mobile_df, Broadband_df):
    df[['Year', 'Quarter']] = df[['Year', 'Quarter']].astype('int64')
    df.sort_values(by=['Year','Quarter'], ascending=True, inplace=True)
